# 06 — QAT Inference (pytorch-quantization INT8)

Loads QAT-trained checkpoints and measures inference time + accuracy on the ImageNet-100 subset.

Currently available: **int8** weights with input quant bits **{8, 4}**.  
Later: int4, int2, int1 weights and TensorRT backend.

In [25]:
from pathlib import Path
import sys, os, json, time

PROJECT_ROOT = Path("..").resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if str(SRC / "qat") not in sys.path:
    sys.path.insert(0, str(SRC / "qat"))

In [26]:
import torch
import torch.nn as nn
import pandas as pd

from config import ExperimentConfig, set_seed
from data import build_runner_loaders
from eval import evaluate
from utils import load_runs, flatten_runs
from qat.quantize import (
    setup_quantization_descriptors,
    initialize_quant_modules,
    get_quantized_model,
)

pd.set_option("display.max_columns", 200)

## 1. Configuration

In [27]:
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints"
RUNS_ROOT       = PROJECT_ROOT / "runs"

NUM_EVAL_BATCHES = 500
BATCH_SIZE       = 1
SEED             = 42
NUM_CLASSES      = 100

# ---- QAT runs to evaluate ----
# Each entry: (model_precision, input_quant_bits, checkpoint_subdir)
# Add new rows here as more QAT checkpoints become available.
QAT_RUNS = [
    ("int8", 8, "qat/resnet18_qat_int8_in8b_cuda_bs256"),
    ("int8", 4, "qat/resnet18_qat_int8_in4b_cuda_bs256"),
    # ("int4", 8, "qat/resnet18_qat_int4_in8b_cuda_bs256"),  # uncomment when available
    # ("int2", 8, "qat/resnet18_qat_int2_in8b_cuda_bs256"),  # uncomment when available
    # ("int1", 8, "qat/resnet18_qat_int1_in8b_cuda_bs256"),  # uncomment when available
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

[Device] cuda


## 2. Quantization setup

Must be called **once** before any model instantiation.

In [28]:
setup_quantization_descriptors()
initialize_quant_modules()
print("[Quant] Descriptors & modules initialized.")

[Quant] Descriptors & modules initialized.


## 3. Helper — run one QAT evaluation

In [29]:
def run_qat_inference(
    model_precision: str,
    input_quant_bits: int,
    checkpoint_subdir: str,
) -> dict:
    """
    Load a QAT checkpoint, evaluate on ImageNet-100 val, and return a
    result payload matching the PTQ JSON schema.
    """
    ckpt_path = str(CHECKPOINT_ROOT / checkpoint_subdir / "qat_phase1_best.pth")
    run_id = f"resnet18_qat_pytorch_{model_precision}_in{input_quant_bits}b_{device.type}_bs{BATCH_SIZE}"

    print(f"\n{'='*70}")
    print(f"[QAT] {run_id}")
    print(f"[QAT] Checkpoint: {ckpt_path}")
    print(f"{'='*70}")

    # --- Config (used for data loading & eval) ---
    cfg = ExperimentConfig(
        backend="pytorch",
        device=device.type,
        batch_size=BATCH_SIZE,
        seed=SEED,
        num_eval_batches=NUM_EVAL_BATCHES,
        input_quant_bits=input_quant_bits,
        model_precision="fp32",        # QAT model runs in fp32 with fake-quant
        num_classes=NUM_CLASSES,
    )
    set_seed(cfg)

    # --- Load QAT model ---
    model = get_quantized_model(ckpt_path, num_classes=NUM_CLASSES)
    model = model.to(device).eval()

    # --- Data ---
    _, loader = build_runner_loaders(cfg)

    # --- Evaluate ---
    criterion = nn.CrossEntropyLoss()
    t0 = time.perf_counter()
    tracker = evaluate(model, loader, cfg, criterion=criterion)
    total_eval_time = round(time.perf_counter() - t0, 3)

    # --- Build result payload (same schema as PTQ) ---
    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": {
            **cfg.to_dict(),
            "backend": "qat_pytorch",
            "model_precision": model_precision,
            "qat_checkpoint": ckpt_path,
        },
        "results": tracker.summary(),
        "error": None,
        "total_eval_time_sec": total_eval_time,
    }
    return payload

## 4. Run all QAT evaluations & save results

In [30]:
records = []

for precision, in_bits, ckpt_dir in QAT_RUNS:
    payload = run_qat_inference(precision, in_bits, ckpt_dir)
    records.append(payload)

    # Save result.json
    out_dir = RUNS_ROOT / payload["run_id"]
    out_dir.mkdir(parents=True, exist_ok=True)
    result_path = out_dir / "result.json"
    with open(result_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
    print(f"[Saved] {result_path}")

for r in records:
    print(f"{r['run_id']}  status={r['status']}  top1={r['results'].get('top1_acc', 'N/A')}")


[QAT] resnet18_qat_pytorch_int8_in8b_cuda_bs1
[QAT] Checkpoint: /home/pf4636/code/resnet/quantized_resnets/checkpoints/qat/resnet18_qat_int8_in8b_cuda_bs256/qat_phase1_best.pth
[data] Filtered to 5000 samples across 100 classes.
Evaluating on 500 batches (first 30 are warmup)...
  --- Warmup complete (30 batches) — starting metric collection ---
  Batch [40/500] Top-1: 90.00% | Top-5: 100.00% | Infer: 29.68 ms/batch
  Batch [50/500] Top-1: 95.00% | Top-5: 100.00% | Infer: 29.24 ms/batch
  Batch [60/500] Top-1: 96.67% | Top-5: 100.00% | Infer: 29.65 ms/batch
  Batch [70/500] Top-1: 97.50% | Top-5: 100.00% | Infer: 29.66 ms/batch
  Batch [80/500] Top-1: 96.00% | Top-5: 100.00% | Infer: 29.92 ms/batch
  Batch [90/500] Top-1: 96.67% | Top-5: 100.00% | Infer: 30.15 ms/batch
  Batch [100/500] Top-1: 95.71% | Top-5: 100.00% | Infer: 29.75 ms/batch
  Batch [110/500] Top-1: 96.25% | Top-5: 100.00% | Infer: 30.10 ms/batch
  Batch [120/500] Top-1: 95.56% | Top-5: 100.00% | Infer: 29.72 ms/batch


## 5. Summary table

In [31]:
runs = load_runs(str(RUNS_ROOT), status="ok")
rows = flatten_runs(runs)
df = pd.DataFrame(rows)

df_qat = df[
    df["cfg.backend"] == "qat_pytorch"
].copy()

df_qat[[
    "run_id",
    "cfg.model_precision",
    "cfg.input_quant_bits",
    "res.top1_acc",
    "res.top5_acc",
    "res.loss_avg",
    "res.infer_ms_avg",
    "res.infer_ms_std",
    "res.throughput_infer_sps",
    "res.total_samples",
]].sort_values("run_id")

,run_id,cfg.model_precision,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.loss_avg,res.infer_ms_avg,res.infer_ms_std,res.throughput_infer_sps,res.total_samples
8,resnet18_qat_pytorch_int8_in4b_cuda_bs1,int8,4,81.702128,96.595745,0.731874,27.327560,4.385585,36.593095,470
9,resnet18_qat_pytorch_int8_in8b_cuda_bs1,int8,8,82.340426,96.170213,0.749352,27.176122,4.437690,36.797009,470


## 6. QAT vs PTQ (FP32 baseline) comparison

In [32]:
# Compare QAT results with PTQ pytorch fp32 baselines at the same input bits
df_ptq_base = df[
    (df["cfg.backend"] == "pytorch")
    & (df["cfg.model_precision"] == "fp32")
    & (df["cfg.device"] == "cuda")
].copy()

compare = pd.concat([
    df_qat[["run_id", "cfg.input_quant_bits", "res.top1_acc", "res.top5_acc", "res.infer_ms_avg"]],
    df_ptq_base[["run_id", "cfg.input_quant_bits", "res.top1_acc", "res.top5_acc", "res.infer_ms_avg"]],
]).sort_values(["cfg.input_quant_bits", "run_id"])

compare

,run_id,cfg.input_quant_bits,res.top1_acc,res.top5_acc,res.infer_ms_avg
4,resnet18_pytorch_fp32_in1b_cuda_bs1,1,26.595745,56.595745,2.869502
5,resnet18_pytorch_fp32_in2b_cuda_bs1,2,49.787234,77.659574,2.818630
6,resnet18_pytorch_fp32_in4b_cuda_bs1,4,80.851064,96.170213,2.698732
8,resnet18_qat_pytorch_int8_in4b_cuda_bs1,4,81.702128,96.595745,27.327560
7,resnet18_pytorch_fp32_in8b_cuda_bs1,8,83.829787,96.595745,3.093748
9,resnet18_qat_pytorch_int8_in8b_cuda_bs1,8,82.340426,96.170213,27.176122
